In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

In [2]:
df = pd.read_csv('dataset.csv')
df

,id,dateadded,url,url_status,last_online,threat,tags,urlhaus_link,reporter
0,3772080,2026-02-04 11:42:17,http://182.113.46.235:50962/bin.sh,online,2026-02-04 11:42:17,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772080/,geenensp
1,3772079,2026-02-04 11:41:08,http://78.29.39.213:40726/i,online,2026-02-04 11:41:08,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772079/,geenensp
2,3772078,2026-02-04 11:34:14,http://182.127.103.194:53383/i,online,2026-02-04 11:34:14,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772078/,geenensp
3,3772077,2026-02-04 11:24:19,http://39.90.179.87:42792/i,online,2026-02-04 11:24:19,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772077/,geenensp
4,3772076,2026-02-04 11:19:11,http://125.44.35.143:37056/i,online,2026-02-04 11:19:11,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772076/,geenensp
...,...,...,...,...,...,...,...,...,...
83415,34178,2018-07-18 22:49:21,http://asl-company.ru/Notification-de-facture-...,online,2026-02-04 06:16:19,malware_download,"doc,emotet,epoch1,heodo",https://urlhaus.abuse.ch/url/34178/,Cryptolaemus1
83416,34102,2018-07-18 18:44:03,http://xn--90abegbttpjb3bzb2j.xn--p1ai/doc/En/...,online,2026-02-04 06:00:25,malware_download,"doc,emotet,heodo",https://urlhaus.abuse.ch/url/34102/,anonymous
83417,28277,2018-07-04 16:45:25,http://crimefreesoftware.com/MC_setup.exe,online,2026-02-04 06:09:00,malware_download,"downloader,exe",https://urlhaus.abuse.ch/url/28277/,lovemalware
83418,16630,2018-06-07 18:40:03,http://robertrowe.com/DOC/Past-Due-invoice/,online,2026-02-04 06:55:17,malware_download,"doc,emotet,epoch1,heodo",https://urlhaus.abuse.ch/url/16630/,Cryptolaemus1


In [3]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.head()

,id,dateadded,url,url_status,last_online,threat,tags,urlhaus_link,reporter
0,3772080,2026-02-04 11:42:17,http://182.113.46.235:50962/bin.sh,online,2026-02-04 11:42:17,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772080/,geenensp
1,3772079,2026-02-04 11:41:08,http://78.29.39.213:40726/i,online,2026-02-04 11:41:08,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772079/,geenensp
2,3772078,2026-02-04 11:34:14,http://182.127.103.194:53383/i,online,2026-02-04 11:34:14,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772078/,geenensp
3,3772077,2026-02-04 11:24:19,http://39.90.179.87:42792/i,online,2026-02-04 11:24:19,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772077/,geenensp
4,3772076,2026-02-04 11:19:11,http://125.44.35.143:37056/i,online,2026-02-04 11:19:11,malware_download,"32-bit,elf,mips,Mozi",https://urlhaus.abuse.ch/url/3772076/,geenensp


In [4]:
offline_df = df[df['url_status']=='offline'].copy() #collecting only offline urls

#incorporating duration
offline_df['date_added'] = pd.to_datetime(offline_df['dateadded'])
offline_df['last_online'] = pd.to_datetime(offline_df['last_online'])
offline_df['duration'] = offline_df['last_online'] - offline_df['date_added']
offline_df['seconds_to_takedown'] = offline_df['duration'].dt.total_seconds()
offline_df = offline_df[offline_df['seconds_to_takedown']>0]

offline_df.head()
offline_df.columns


Index(['id', 'dateadded', 'url', 'url_status', 'last_online', 'threat', 'tags',
       'urlhaus_link', 'reporter', 'date_added', 'duration',
       'seconds_to_takedown'],
      dtype='object')

In [5]:
#removing unnecessary columns
offline_df.drop(columns=['urlhaus_link','reporter','url_status','threat'],inplace=True)
offline_df.head()

,id,dateadded,url,last_online,tags,date_added,duration,seconds_to_takedown
239,3771839,2026-02-03 23:05:09,http://130.12.180.43/files/8428202012/bIMApgG.exe,2026-02-04 06:06:29,"c2-monitor-auto,dropped-by-amadey,Smoke Loader",2026-02-03 23:05:09,0 days 07:01:20,25280.0
244,3771834,2026-02-03 22:27:20,http://123.13.109.106:35850/bin.sh,2026-02-03 23:54:21,"32-bit,elf,mips,Mozi",2026-02-03 22:27:20,0 days 01:27:01,5221.0
258,3771820,2026-02-03 21:03:15,http://175.165.84.104:49093/i,2026-02-04 00:02:26,"32-bit,elf,Mozi",2026-02-03 21:03:15,0 days 02:59:11,10751.0
262,3771816,2026-02-03 21:02:15,http://110.37.78.200:49619/i,2026-02-04 01:05:01,"32-bit,elf,Mozi",2026-02-03 21:02:15,0 days 04:02:46,14566.0
263,3771815,2026-02-03 21:02:14,http://179.126.29.17:8081/Photo.scr,2026-02-03 23:59:51,CoinMiner,2026-02-03 21:02:14,0 days 02:57:37,10657.0


In [6]:
#adding url structure features
import re

def find_length(url):
    return len(url)

offline_df['url_length'] = offline_df['url'].apply(find_length)

def is_https(url):
    if url[:5]=='https':
        return 1
    else:
        return 0
    
offline_df['is_https'] = offline_df['url'].apply(is_https)

def path_depth(url):
    url_split = '//'.join(url.split('//')[1:]) #catering for cases where multiple // might exist
    return len([p for p in url_split.split('/') if p])-1 #not counting the ip itself

offline_df['path_depth'] = offline_df['url'].apply(path_depth)

def is_ip(url):
    pattern = r'\d+\.\d+\.\d+\.\d+'
    if len(re.findall(pattern,url))>0:
        return 1
    else:
        return 0
    
offline_df['is_ip'] = offline_df['url'].apply(is_ip)

def file_extension(url):
    if '.' in url.split('/')[-1]:
        return url.split('.')[-1]
    else:
        return 'none'
    
offline_df['file_extension']=offline_df['url'].apply(file_extension)

offline_df.head()



,id,dateadded,url,last_online,tags,date_added,duration,seconds_to_takedown,url_length,is_https,path_depth,is_ip,file_extension
239,3771839,2026-02-03 23:05:09,http://130.12.180.43/files/8428202012/bIMApgG.exe,2026-02-04 06:06:29,"c2-monitor-auto,dropped-by-amadey,Smoke Loader",2026-02-03 23:05:09,0 days 07:01:20,25280.0,49,0,3,1,exe
244,3771834,2026-02-03 22:27:20,http://123.13.109.106:35850/bin.sh,2026-02-03 23:54:21,"32-bit,elf,mips,Mozi",2026-02-03 22:27:20,0 days 01:27:01,5221.0,34,0,1,1,sh
258,3771820,2026-02-03 21:03:15,http://175.165.84.104:49093/i,2026-02-04 00:02:26,"32-bit,elf,Mozi",2026-02-03 21:03:15,0 days 02:59:11,10751.0,29,0,1,1,none
262,3771816,2026-02-03 21:02:15,http://110.37.78.200:49619/i,2026-02-04 01:05:01,"32-bit,elf,Mozi",2026-02-03 21:02:15,0 days 04:02:46,14566.0,28,0,1,1,none
263,3771815,2026-02-03 21:02:14,http://179.126.29.17:8081/Photo.scr,2026-02-03 23:59:51,CoinMiner,2026-02-03 21:02:14,0 days 02:57:37,10657.0,35,0,1,1,scr


In [7]:
#adding temporal feature - day of the week
offline_df.drop(columns='dateadded',inplace=True)

offline_df['day_of_week'] = offline_df['date_added'].dt.day_name()
offline_df.head()


,id,url,last_online,tags,date_added,duration,seconds_to_takedown,url_length,is_https,path_depth,is_ip,file_extension,day_of_week
239,3771839,http://130.12.180.43/files/8428202012/bIMApgG.exe,2026-02-04 06:06:29,"c2-monitor-auto,dropped-by-amadey,Smoke Loader",2026-02-03 23:05:09,0 days 07:01:20,25280.0,49,0,3,1,exe,Tuesday
244,3771834,http://123.13.109.106:35850/bin.sh,2026-02-03 23:54:21,"32-bit,elf,mips,Mozi",2026-02-03 22:27:20,0 days 01:27:01,5221.0,34,0,1,1,sh,Tuesday
258,3771820,http://175.165.84.104:49093/i,2026-02-04 00:02:26,"32-bit,elf,Mozi",2026-02-03 21:03:15,0 days 02:59:11,10751.0,29,0,1,1,none,Tuesday
262,3771816,http://110.37.78.200:49619/i,2026-02-04 01:05:01,"32-bit,elf,Mozi",2026-02-03 21:02:15,0 days 04:02:46,14566.0,28,0,1,1,none,Tuesday
263,3771815,http://179.126.29.17:8081/Photo.scr,2026-02-03 23:59:51,CoinMiner,2026-02-03 21:02:14,0 days 02:57:37,10657.0,35,0,1,1,scr,Tuesday


In [8]:
#adding tag based features

tags_dict = {"architecture_tags" :['elf', '32-bit', '64-bit', 'mips', 'arm', 'x86', 'x86-64', 'PowerPC', 'm68k', 'SuperH', 'sparc', 'AArch64', 'RISC-V', 'macho', 'android', 'macOS', 'windows', 'linux'],
'filetype_tags' :['exe', 'zip', 'apk', 'pdf', 'doc', 'msi', 'dll', 'lnk', 'iso', 'rar', 'xml', 'jar', 'svg', 'xls', 'cab', 'tar', 'gz', 'tgz', '7z', 'scr', 'jse', 'wsf'],
'botnet_iot_tags' : ['Mozi', 'mirai', 'gafgyt', 'Kaiji', 'hajime', 'MyloBot', 'Tsunami', 'Ngioweb', 'HailBot', 'P2Pinfect', 'Xorddos', 'aisuru', 'moobot', 'GoBrut', 'PumaBot', 'VioletWorm', 'iranbot'],
'stealer_malware_tags' : ['Vidar', 'RedLineStealer', 'LummaStealer', 'AgentTesla', 'FormBook', 'Stealc', 'MaskGramStealer', 'SalatStealer', 'StrelaStealer', 'PhantomStealer', 'PureLogsStealer', 'rustystealer', 'SnakeKeylogger', 'ArkeiStealer', 'VIPKeylogger', 'ACRStealer', 'BansheeStealer', 'MeduzaStealer'],
'rat_tags': ['RemcosRAT', 'AsyncRAT', 'QuasarRAT', 'njRAT', 'worm', 'NanoCore', 'NetSupportManager' 'RAT', 'DarkVisionRAT', 'ValleyRAT', 'ClayRAT', 'L3mon', 'OctoRAT', 'Gh0stRAT', 'SparkRAT', 'PupyRAT', 'QatarRAT'],
'loaders_droppers_tags' : ['SmartLoader', 'GuLoader', 'donutloader', 'HijackLoader', 'Amadey', 'PrivateLoader', 'DBatLoader', 'KoiLoader', 'T34loader', 'GodLoader', 'CaminhoLoader', 'OffLoader', 'DarkTortilla', 'rev-base64-loader', 'base64-loader', 'js-GhoLoader','Smoke Loader'],
'c2_tags' : ['CobaltStrike', 'Sliver', 'Metasploit', 'Havoc', 'Merlin', 'Koadic', 'meterpreter', 'Pyramid C2', 'AdaptixC2', 'Ligolo', 'supershell-c2', 'bruteratel', 'Covenant'],
'encoding_tags': ['ascii', 'encrypted', 'Encoded', 'base64', 'hex', 'obfuscated', 'stego', 'msi-stego', 'rev-base64-loader', 'xor', 'Password-protected', 'AES', 'stego-london'],
'web_delivery_tags':['powershell', 'ps1', 'js', 'html', 'iframe', 'vbs', 'bat', 'sh', 'script', 'hta', 'AppleScript', 'lua', 'perl', 'python', 'php', 'bash'],
'ransomware_tags':['lockbit', 'wannacry', 'Ransomware', 'GandCrab', 'Dharma', 'Hive', 'Jigsaw', 'petya', 'BlackMatter', 'RedLocker', 'Troldesh', 'Jaff', 'CryptoLocker'],
'coinminer_tags':['CoinMiner', 'xmrig', 'cryptojacking', 'monero', 'coinhive', 'redtail', 'kinsing', 'CryptoMiner', 'miner'],
'social_engineering_tags':['ClearFake', 'FakeCaptcha', 'ClickFix', 'SmartApeSG', 'SocGholish', 'seopoisoning', 'FakeCheat', 'Fake OS Update', 'fake-git', 'fake-DHL', 'fakeupdates', 'phishing', 'FakeWallet'],
'remote_tool_tags':['connectwise', 'screenconnect', 'GoToResolve', 'NetSupport', 'SyncroRMM', 'FleetDeck', 'MeshAgent', 'DattoRMM', 'N-able', 'SimpleHelp', 'AteraAgent', 'PDQConnect'],
'detection_tags':['c2-monitor-auto', 'censys', 'huntio', 'Netskope', 'botnetdomain', 'opendir', 'geofenced', 'generic-av-detection', '10pluspositivesinVT', 'honeypot', 'urlquery'],
'delivery_method_tags':['ua-wget', 'ua-curl', 'ua-android', 'ua-powershell', 'tftp', 'wget', 'dropped-by-amadey', 'dropped-by-Phorpiex', 'dropped-by-Stealc', 'spammed-by-tofsee', 'trycloudflare', 'github', 'gitlab', 'bitbucket'],
'geography_tags':['USA', 'BRA', 'DEU', 'MEX', 'ITA', 'CHE', 'KOR', 'NLD', 'SVK', 'ITA']}

def malicious_family_finder(tags):
    splitted = tags.split(',')
    family = []
    for elem in splitted:
        if elem in tags_dict['botnet_iot_tags']:
            if "IoT Botnet" not in family:
                family.append("IoT Botnet")
        elif elem in tags_dict['stealer_malware_tags']:
            if "Stealer Malware" not in family:
                family.append("Stealer Malware")
        elif elem in tags_dict['rat_tags']:
            if "Remote Access Trojan (RAT)" not in family:
                family.append("Remote Access Trojan (RAT)")
        elif elem in tags_dict['loaders_droppers_tags']:
            if "Loader/Dropper" not in family:
                family.append("Loader/Dropper")
        elif elem in tags_dict['coinminer_tags']:
            if "Coin Miner" not in family:
                family.append('Coin Miner')
        elif elem in tags_dict['social_engineering_tags']:
            if "Social Engineering Attack" not in family:
                family.append('Social Engineering Attack')
        else:
            continue

    return family
    
offline_df['Malware Families']=offline_df['tags'].apply(malicious_family_finder)


def architecture_finder(tags):
    splitted = tags.split(',')
    architecture_tags=[]
    for elem in splitted:
        if elem in tags_dict['architecture_tags']:
            if elem not in architecture_tags:
                architecture_tags.append(elem)
        
    return architecture_tags

offline_df['Architecture'] = offline_df['tags'].apply(architecture_finder)

# def file_type_finder(tags):
#     splitted = tags.split(',')
#     file_tags = []
#     for elem in splitted:
#         if elem in tags_dict['filetype_tags']:
#             if elem not in file_tags:
#                 file_tags.append(elem)
#     return file_tags

# offline_df['File Type']=offline_df['tags'].apply(file_type_finder)

offline_df.head()

,id,url,last_online,tags,date_added,duration,seconds_to_takedown,url_length,is_https,path_depth,is_ip,file_extension,day_of_week,Malware Families,Architecture
239,3771839,http://130.12.180.43/files/8428202012/bIMApgG.exe,2026-02-04 06:06:29,"c2-monitor-auto,dropped-by-amadey,Smoke Loader",2026-02-03 23:05:09,0 days 07:01:20,25280.0,49,0,3,1,exe,Tuesday,[Loader/Dropper],[]
244,3771834,http://123.13.109.106:35850/bin.sh,2026-02-03 23:54:21,"32-bit,elf,mips,Mozi",2026-02-03 22:27:20,0 days 01:27:01,5221.0,34,0,1,1,sh,Tuesday,[IoT Botnet],"[32-bit, elf, mips]"
258,3771820,http://175.165.84.104:49093/i,2026-02-04 00:02:26,"32-bit,elf,Mozi",2026-02-03 21:03:15,0 days 02:59:11,10751.0,29,0,1,1,none,Tuesday,[IoT Botnet],"[32-bit, elf]"
262,3771816,http://110.37.78.200:49619/i,2026-02-04 01:05:01,"32-bit,elf,Mozi",2026-02-03 21:02:15,0 days 04:02:46,14566.0,28,0,1,1,none,Tuesday,[IoT Botnet],"[32-bit, elf]"
263,3771815,http://179.126.29.17:8081/Photo.scr,2026-02-03 23:59:51,CoinMiner,2026-02-03 21:02:14,0 days 02:57:37,10657.0,35,0,1,1,scr,Tuesday,[Coin Miner],[]


In [9]:
#adding port features
def port(url):
    is_https = False
    if url[:5]=='https':
        is_https=True
    url_split = '//'.join(url.split('//')[1:])
    host = url_split.split('/')[0]  #seperating the host:path
    if ':' in host:
        return int(host.split(':')[1])
    else:
        if is_https:
            return 443
        else:
            return 80
        
offline_df['port'] = offline_df['url'].apply(port)
offline_df.head()


,id,url,last_online,tags,date_added,duration,seconds_to_takedown,url_length,is_https,path_depth,is_ip,file_extension,day_of_week,Malware Families,Architecture,port
239,3771839,http://130.12.180.43/files/8428202012/bIMApgG.exe,2026-02-04 06:06:29,"c2-monitor-auto,dropped-by-amadey,Smoke Loader",2026-02-03 23:05:09,0 days 07:01:20,25280.0,49,0,3,1,exe,Tuesday,[Loader/Dropper],[],80
244,3771834,http://123.13.109.106:35850/bin.sh,2026-02-03 23:54:21,"32-bit,elf,mips,Mozi",2026-02-03 22:27:20,0 days 01:27:01,5221.0,34,0,1,1,sh,Tuesday,[IoT Botnet],"[32-bit, elf, mips]",35850
258,3771820,http://175.165.84.104:49093/i,2026-02-04 00:02:26,"32-bit,elf,Mozi",2026-02-03 21:03:15,0 days 02:59:11,10751.0,29,0,1,1,none,Tuesday,[IoT Botnet],"[32-bit, elf]",49093
262,3771816,http://110.37.78.200:49619/i,2026-02-04 01:05:01,"32-bit,elf,Mozi",2026-02-03 21:02:15,0 days 04:02:46,14566.0,28,0,1,1,none,Tuesday,[IoT Botnet],"[32-bit, elf]",49619
263,3771815,http://179.126.29.17:8081/Photo.scr,2026-02-03 23:59:51,CoinMiner,2026-02-03 21:02:14,0 days 02:57:37,10657.0,35,0,1,1,scr,Tuesday,[Coin Miner],[],8081


In [10]:
#collecting useful features
features = offline_df.drop(columns=['id','url','date_added'])
features


,last_online,tags,duration,seconds_to_takedown,url_length,is_https,path_depth,is_ip,file_extension,day_of_week,Malware Families,Architecture,port
239,2026-02-04 06:06:29,"c2-monitor-auto,dropped-by-amadey,Smoke Loader",0 days 07:01:20,25280.0,49,0,3,1,exe,Tuesday,[Loader/Dropper],[],80
244,2026-02-03 23:54:21,"32-bit,elf,mips,Mozi",0 days 01:27:01,5221.0,34,0,1,1,sh,Tuesday,[IoT Botnet],"[32-bit, elf, mips]",35850
258,2026-02-04 00:02:26,"32-bit,elf,Mozi",0 days 02:59:11,10751.0,29,0,1,1,none,Tuesday,[IoT Botnet],"[32-bit, elf]",49093
262,2026-02-04 01:05:01,"32-bit,elf,Mozi",0 days 04:02:46,14566.0,28,0,1,1,none,Tuesday,[IoT Botnet],"[32-bit, elf]",49619
263,2026-02-03 23:59:51,CoinMiner,0 days 02:57:37,10657.0,35,0,1,1,scr,Tuesday,[Coin Miner],[],8081
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73746,2025-11-20 21:21:54,"CoinMiner,opendir",14 days 09:03:47,1242227.0,202,0,4,1,scr,Thursday,[Coin Miner],[],6868
73747,2025-11-20 22:13:44,"CoinMiner,opendir",14 days 09:55:38,1245338.0,158,0,3,1,scr,Thursday,[Coin Miner],[],6868
73751,2025-11-07 06:51:33,"32-bit,elf,mips,Mozi",0 days 18:34:15,66855.0,33,0,1,1,sh,Thursday,[IoT Botnet],"[32-bit, elf, mips]",37823
73755,2025-11-08 05:25:04,"32-bit,elf,mips,Mozi",1 days 17:23:52,149032.0,32,0,1,1,sh,Thursday,[IoT Botnet],"[32-bit, elf, mips]",38770


In [15]:
offline_df.columns

Index(['id', 'url', 'last_online', 'tags', 'date_added', 'duration',
       'seconds_to_takedown', 'url_length', 'is_https', 'path_depth', 'is_ip',
       'file_extension', 'day_of_week', 'Malware Families', 'Architecture',
       'port'],
      dtype='object')

In [22]:
offline_df['Malware Families'].value_counts()

Malware Families
[IoT Botnet]                                     33992
[Coin Miner]                                      3145
[]                                                3119
[Stealer Malware]                                  364
[Social Engineering Attack]                        308
[Loader/Dropper]                                   204
[Remote Access Trojan (RAT)]                       155
[Loader/Dropper, Stealer Malware]                   23
[Coin Miner, IoT Botnet]                            15
[Loader/Dropper, Remote Access Trojan (RAT)]        12
[Stealer Malware, Loader/Dropper]                   11
[Remote Access Trojan (RAT), Loader/Dropper]         5
[Coin Miner, Stealer Malware]                        3
[Social Engineering Attack, Loader/Dropper]          2
[Stealer Malware, Remote Access Trojan (RAT)]        1
[Coin Miner, Remote Access Trojan (RAT)]             1
Name: count, dtype: int64